In [6]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import operator

# Core LangGraph components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

# LLM and messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# Load API keys and set up tracing
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Intro to LangGraph"

# Define the State
class GraphState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Define Nodes
def call_llm(state: GraphState):
    print("--- Calling LLM ---")
    llm = ChatOpenAI(model="gpt-4o")
    response = llm.invoke(state['messages'])
    return {"messages": [response]}

# Build the Graph (Simple chatbot structure)
workflow = StateGraph(GraphState)
workflow.add_node("llm", call_llm)
workflow.add_edge(START, "llm")
workflow.add_edge("llm", END)

# Use SqliteSaver as a context manager
with SqliteSaver.from_conn_string(":memory:") as memory:

    # Compile the graph with the memory saver
    app = workflow.compile(checkpointer=memory)

    # --- Run a few turns of conversation ---
    config = {"configurable": {"thread_id": "time-travel-thread-1"}}
    saved_state_config = None # To store the config of the state we want to modify

    print("--- Turn 1 ---")
    turn1_input = {"messages": [HumanMessage(content="Hi! My name is Bob.")]}
    # Use invoke() for the first turn to ensure it completes
    state_after_turn1 = app.invoke(turn1_input, config)
    state_after_turn1["messages"][-1].pretty_print()

    # Get the config from the state history *after* the invoke
    history = list(app.get_state_history(config)) # Convert generator to list

    if history:
        saved_state_config = history[-1].config # Get the config of the last saved state
        print(f"\nConfig captured for state after Turn 1: {saved_state_config}")
    else:
        print("Error: Could not retrieve state history.")

    # Proceed only if we successfully got the config
    if saved_state_config:
        print("\n--- Turn 2 ---")
        turn2_input = {"messages": [HumanMessage(content="What's my name?")]}
        state_after_turn2 = app.invoke(turn2_input, config)
        state_after_turn2["messages"][-1].pretty_print()

        # --- Time Travel: Modify a Past State ---
        print(f"\n--- Time Traveling back using saved config ---")

        # Get the state *as it was* using the saved config
        past_state = app.get_state(saved_state_config)

        print("State Before Turn 2:")
        for msg in past_state.values['messages']:
            print(f"- {msg.type}: {msg.content}")

        # Modify this past state
        modified_messages = list(past_state.values['messages'])

        # FIX: Explicitly target the first message (index 0)
        if len(modified_messages) > 0:
            modified_messages[0] = HumanMessage(content="Hi! My name is Alice.")
        else:
            print("Warning: Could not modify past state as message list was unexpectedly empty.")


        # Update the state *at that specific point in history* using the saved config
        app.update_state(saved_state_config, {"messages": modified_messages})
        print("\nState modified in the past.")


        # --- Continue from the Modified Past State ---
        print("\n--- Rerunning Turn 2 from the modified history ---")
        turn2_rerun_input = {"messages": [HumanMessage(content="What's my name?")]}
        # LangGraph automatically uses the latest state for the thread
        state_after_rerun = app.invoke(turn2_rerun_input, config)
        print("AI Response (after time travel):")
        state_after_rerun["messages"][-1].pretty_print()
    else:
        print("\nSkipping Time Travel steps due to missing config.")

--- Turn 1 ---
--- Calling LLM ---
================================== Ai Message ==================================

Hello Bob! How can I assist you today?

Config captured for state after Turn 1: {'configurable': {'thread_id': 'time-travel-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b3233-73f1-6b9d-bfff-bd4d7521ef44'}}

--- Turn 2 ---
--- Calling LLM ---
================================== Ai Message ==================================

Your name is Bob.

--- Time Traveling back using saved config ---
State Before Turn 2:

State modified in the past.

--- Rerunning Turn 2 from the modified history ---
--- Calling LLM ---
AI Response (after time travel):
================================== Ai Message ==================================

Your name is Bob.
